[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# JSON on Disk &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.


In [1]:

import json
import datetime
from pathlib import Path
import shutil

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)


**1.** Write a list of two dictionaries to `scratch/people.json` with `indent=2`, then print
the file text.


In [2]:

people = [
    {"name": "Ada", "role": "analyst"},
    {"name": "Grace", "role": "engineer"},
]

target = scratch / "people.json"

with open(target, "w", encoding="utf-8") as f:
    json.dump(people, f, indent=2)

print(target.read_text(encoding="utf-8"))


[
  {
    "name": "Ada",
    "role": "analyst"
  },
  {
    "name": "Grace",
    "role": "engineer"
  }
]


**2.** Load it back and confirm it equals what you wrote.


In [3]:

with open(target, encoding="utf-8") as f:
    loaded = json.load(f)

print(loaded)
print("identical:", loaded == people)


[{'name': 'Ada', 'role': 'analyst'}, {'name': 'Grace', 'role': 'engineer'}]
identical: True


This round trip is exact because every value was already a JSON type: strings inside
dictionaries inside a list. The next task is the case where that is not true.


**3.** Round-trip `{"point": (3, 7), "ids": {10: "a"}}` and print the result.


In [4]:

original = {"point": (3, 7), "ids": {10: "a"}}
restored = json.loads(json.dumps(original))

print("in: ", original)
print("out:", restored)
print("equal:", original == restored)

# The tuple (3, 7) became the list [3, 7], and the integer key 10 became the string "10".
# Neither change was reported.


in:  {'point': (3, 7), 'ids': {10: 'a'}}
out: {'point': [3, 7], 'ids': {'10': 'a'}}
equal: False


`restored["ids"][10]` would now raise a `KeyError` on data that visibly contains a `10`. That is
the failure this change produces, and it happens far from the line that caused it.


**4.** Try to write `{"seen": {1, 2, 3}}` and print the exception, then write it
successfully.


In [5]:

try:
    json.dumps({"seen": {1, 2, 3}})
except TypeError as e:
    print(type(e).__name__, "-", e)

print(json.dumps({"seen": sorted({1, 2, 3})}))


TypeError - Object of type set is not JSON serializable
{"seen": [1, 2, 3]}


`sorted` was a decision, not a formality. A set has no order, so writing one to JSON means
choosing an order, and sorting makes the output stable between runs.


**5.** Write `{"today": ..., "note": "x"}` using a `default=` function, then load it and
convert the date back.


In [6]:

def encode_extra(obj):
    if isinstance(obj, (datetime.date, datetime.datetime)):
        return obj.isoformat()
    raise TypeError(f"cannot serialize {type(obj).__name__}")


payload = {"today": datetime.date(2026, 3, 1), "note": "x"}
text = json.dumps(payload, default=encode_extra)

print("written:", text)

restored = json.loads(text)
print("loaded: ", restored, "| today is a", type(restored["today"]).__name__)

as_date = datetime.date.fromisoformat(restored["today"])
print("converted back:", as_date, type(as_date).__name__)


written: {"today": "2026-03-01", "note": "x"}
loaded:  {'today': '2026-03-01', 'note': 'x'} | today is a str
converted back: 2026-03-01 date


Three steps, and the third is the one people forget. `default=` gets the date out; nothing gets
it back in automatically, because the file has no way to say that string was a date.


**6.** Write three records as JSON Lines, then read the file back one line at a time.


In [7]:

events = scratch / "events.jsonl"

records = [
    {"name": "Ada", "score": 91},
    {"name": "Grace", "score": 88},
    {"name": "Alan", "score": 95},
]

with open(events, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record) + "\n")

with open(events, encoding="utf-8") as f:
    for line in f:
        print(json.loads(line)["name"])


Ada
Grace
Alan


The `+ "\n"` matters. Without it the whole file is one line and neither format works: it is not
valid JSON, and it is not JSON Lines either.

Only one record is in memory at any moment, which is the reason to use this shape at all.


In [8]:

shutil.rmtree(scratch)

print("cleaned up:", not scratch.exists())


cleaned up: True


---

&#8592; **Back to:** [JSON on Disk](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/05-json-on-disk.ipynb)  &nbsp;&middot;&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)
